In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)


# ------------------------------------------------------------
# 2. DOWNLOAD THE EXACT FREECODECAMP DATASET
# ------------------------------------------------------------

train_url = "https://cdn.freecodecamp.org/project-data/sms/train-data.tsv"
test_url = "https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv"

train_file = "train-data.tsv"
test_file = "valid-data.tsv"

# Download only if files do not already exist
if not os.path.exists(train_file):
    tf.keras.utils.get_file(
        train_file,
        train_url,
        cache_dir=".",
        cache_subdir=""
    )

if not os.path.exists(test_file):
    tf.keras.utils.get_file(
        test_file,
        test_url,
        cache_dir=".",
        cache_subdir=""
    )


# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

train_data = pd.read_csv(
    train_file,
    sep="\t",
    header=None,
    names=["label", "text"]
)

test_data = pd.read_csv(
    test_file,
    sep="\t",
    header=None,
    names=["label", "text"]
)

print("\nDataset loaded successfully.")

print("Training data:", train_data.shape)
print("Testing data :", test_data.shape)


# ------------------------------------------------------------
# 4. CONVERT LABELS
# ------------------------------------------------------------

# ham  = 0
# spam = 1

train_data["label"] = train_data["label"].map({
    "ham": 0,
    "spam": 1
})

test_data["label"] = test_data["label"].map({
    "ham": 0,
    "spam": 1
})


# Make sure all messages are strings
train_data["text"] = train_data["text"].astype(str)
test_data["text"] = test_data["text"].astype(str)


# ------------------------------------------------------------
# 5. PREPARE INPUTS
# ------------------------------------------------------------

X_train = train_data["text"].values
y_train = train_data["label"].values

X_test = test_data["text"].values
y_test = test_data["label"].values

print("\nTraining messages:", len(X_train))
print("Testing messages :", len(X_test))


# ------------------------------------------------------------
# 6. TEXT VECTORIZATION
# ------------------------------------------------------------

MAX_FEATURES = 10000
SEQUENCE_LENGTH = 100

vectorizer = layers.TextVectorization(
    max_tokens=MAX_FEATURES,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation"
)

# Learn vocabulary from training messages
vectorizer.adapt(X_train)

print(
    "Vocabulary size:",
    len(vectorizer.get_vocabulary())
)


# ------------------------------------------------------------
# 7. CREATE NEURAL NETWORK
# ------------------------------------------------------------

model = keras.Sequential([

    # Text -> integer sequences
    vectorizer,

    # Word embeddings
    layers.Embedding(
        input_dim=MAX_FEATURES,
        output_dim=64
    ),

    # Learn sequence patterns
    layers.Bidirectional(
        layers.LSTM(64)
    ),

    # Fully connected layer
    layers.Dense(
        64,
        activation="relu"
    ),

    # Reduce overfitting
    layers.Dropout(0.5),

    # Binary classification
    # 0 = ham
    # 1 = spam
    layers.Dense(
        1,
        activation="sigmoid"
    )
])


# ------------------------------------------------------------
# 8. COMPILE MODEL
# ------------------------------------------------------------

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nModel created successfully.")


# ------------------------------------------------------------
# 9. TRAIN MODEL
# ------------------------------------------------------------

print("\nStarting training...")
print("=" * 60)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32,
    verbose=1
)


# ------------------------------------------------------------
# 10. EVALUATE MODEL
# ------------------------------------------------------------

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\n" + "=" * 60)
print("MODEL RESULTS")
print("=" * 60)

print(
    "Validation accuracy:",
    round(accuracy * 100, 2),
    "%"
)

print(
    "Validation loss:",
    round(loss, 4)
)


# ------------------------------------------------------------
# 11. REQUIRED freeCodeCamp FUNCTION
# ------------------------------------------------------------

def predict_message(pred_text):

    # Convert input to string
    pred_text = str(pred_text)

    # Predict probability of spam
    spam_probability = float(
        model.predict(
            tf.constant([pred_text]), # Changed np.array([pred_text]) to tf.constant([pred_text])
            verbose=0
        )[0][0]
    )

    # Determine prediction
    if spam_probability >= 0.5:
        prediction = "spam"
        probability = spam_probability

    else:
        prediction = "ham"
        probability = 1 - spam_probability

    return [
        probability,
        prediction
    ]


# ------------------------------------------------------------
# 12. TEST REQUIRED FUNCTION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TESTING predict_message()")
print("=" * 60)

messages = [

    "how are you doing today",

    "sale today! to stop texts call 98912460324",

    "i dont want to go. can we try it a different day? available sat",

    "our new mobile video service is live. just install on your phone to start watching.",

    "you have won £1000 cash! call to claim your prize.",

    "i'll bring it tomorrow. don't forget the milk.",

    "wow, is your arm alright. that happened to me one time too"

]

for message in messages:

    result = predict_message(message)

    print("\nMessage:")
    print(message)

    print("Prediction:")
    print(result)


# ------------------------------------------------------------
# 13. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PROJECT COMPLETE")
print("=" * 60)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

print("Function: predict_message(message)")
print("Output: [probability, 'ham' or 'spam']")

if accuracy >= 0.95:
    print("\nHigh accuracy achieved.")

print("\nNow run the ORIGINAL freeCodeCamp final test cell.")

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):

    # Convert input to string
    pred_text = str(pred_text)

    # Predict probability of spam
    spam_probability = float(
        model.predict(
            tf.constant([pred_text]),
            verbose=0
        )[0][0]
    )

    # Determine prediction
    if spam_probability >= 0.5:
        prediction = "spam"
        probability = spam_probability

    else:
        prediction = "ham"
        probability = 1 - spam_probability

    return [
        probability,
        prediction
    ]

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
